In [ ]:
import operator
from typing import Annotated, Any, Dict, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class ProcessState(TypedDict):
    current_node: str
    passed_nodes: Annotated[list[str], operator.add]


def process_data_node(state: ProcessState) -> Dict[str, Any]:
    node_name = "数据处理"
    step_info = f"{node_name}: 加载并验证数据"
    return {"current_node": node_name, "passed_nodes": [step_info]}


def analyze_data_node(state: ProcessState) -> Dict[str, Any]:
    node_name = "数据分析"
    step_info = f"{node_name}: 生成分析报告"
    return {"current_node": node_name, "passed_nodes": [step_info]}


workflow = StateGraph(ProcessState)
workflow.add_node("process_data_node", process_data_node)
workflow.add_node("analyze_data_node", analyze_data_node)

workflow.add_edge(START, "process_data_node")
workflow.add_edge("process_data_node", "analyze_data_node")
workflow.add_edge("analyze_data_node", END)

chechpointer = InMemorySaver()
graph = workflow.compile(checkpointer=chechpointer)


import json

from rich.console import Console
from rich.panel import Panel

console = Console()

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

initial_state = graph.get_state(config)
console.print(Panel(json.dumps(initial_state, indent=4), title="初始状态"))


graph.invoke({"current_node": ""}, config)
final_state = graph.get_state(config)
initial_state = graph.get_state(config)
console.print(Panel(json.dumps(final_state, indent=4), title="最终状态"))

state_history = list(graph.get_state_history(config))
state_history = [json.dumps(x, indent=4) for x in state_history]
console.print(Panel(f"{len(state_history)}", title="history"))

graph.update_state(config, {"current_node": "test", "passed_nodes": ["test"]})
test_state = graph.get_state(config)
console.print(Panel(json.dumps(test_state, indent=4), title="test"))

state_history = list(graph.get_state_history(config))
state_history = [json.dumps(x, indent=4) for x in state_history]
console.print(Panel(f"{len(state_history)}", title="history"))


╭─────────────────────────────────────────────────── 初始状态 ────────────────────────────────────────────────────╮
│ [                                                                                                               │
│     {},                                                                                                         │
│     [],                                                                                                         │
│     {                                                                                                           │
│         "configurable": {                                                                                       │
│             "thread_id": "1"                                                                                    │
│         }                                                                                                       │
│     },                                                                                                          │
│     null,                                                                                                       │
│     null,                                                                                                       │
│     null,                                                                                                       │
│     [],                                                                                                         │
│     []                                                                                                          │
│ ]                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 最终状态 ────────────────────────────────────────────────────╮
│ [                                                                                                               │
│     {                                                                                                           │
│         "current_node": "\u6570\u636e\u5206\u6790",                                                             │
│         "passed_nodes": [                                                                                       │
│             "\u6570\u636e\u5904\u7406: \u52a0\u8f7d\u5e76\u9a8c\u8bc1\u6570\u636e",                             │
│             "\u6570\u636e\u5206\u6790: \u751f\u6210\u5206\u6790\u62a5\u544a"                                    │
│         ]                                                                                                       │
│     },                                                                                                          │
│     [],                                                                                                         │
│     {                                                                                                           │
│         "configurable": {                                                                                       │
│             "thread_id": "1",                                                                                   │
│             "checkpoint_ns": "",                                                                                │
│             "checkpoint_id": "1f178796-c9c7-6f7e-8002-2070492870e8"                                             │
│         }                                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│         "source": "loop",                                                                                       │
│         "step": 2,                                                                                              │
│         "parents": {}                                                                                           │
│     },                                                                                                          │
│     "2026-07-05T13:57:13.376749+00:00",                                                                         │
│     {                                                                                                           │
│         "configurable": {                                                                                       │
│             "thread_id": "1",                                                                                   │
│             "checkpoint_ns": "",                                                                                │
│             "checkpoint_id": "1f178796-c9c6-6fde-8001-90b97e7a0c63"                                             │
│         }                                                                                                       │
│     },                                                                                                          │
│     [],                                                                                                         │
│     []                                                                                                          │
│ ]                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── history ────────────────────────────────────────────────────╮
│ 4                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── test ──────────────────────────────────────────────────────╮
│ [                                                                                                               │
│     {                                                                                                           │
│         "current_node": "test",                                                                                 │
│         "passed_nodes": [                                                                                       │
│             "\u6570\u636e\u5904\u7406: \u52a0\u8f7d\u5e76\u9a8c\u8bc1\u6570\u636e",                             │
│             "\u6570\u636e\u5206\u6790: \u751f\u6210\u5206\u6790\u62a5\u544a",                                   │
│             "test"                                                                                              │
│         ]                                                                                                       │
│     },                                                                                                          │
│     [],                                                                                                         │
│     {                                                                                                           │
│         "configurable": {                                                                                       │
│             "thread_id": "1",                                                                                   │
│             "checkpoint_ns": "",                                                                                │
│             "checkpoint_id": "1f178796-c9d0-673c-8003-d907497389e5"                                             │
│         }                                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│         "source": "update",                                                                                     │
│         "step": 3,                                                                                              │
│         "parents": {}                                                                                           │
│     },                                                                                                          │
│     "2026-07-05T13:57:13.380224+00:00",                                                                         │
│     {                                                                                                           │
│         "configurable": {                                                                                       │
│             "thread_id": "1",                                                                                   │
│             "checkpoint_ns": "",                                                                                │
│             "checkpoint_id": "1f178796-c9c7-6f7e-8002-2070492870e8"                                             │
│         }                                                                                                       │
│     },                                                                                                          │
│     [],                                                                                                         │
│     []                                                                                                          │
│ ]                                                                                                               │
╰───────────────────────────────────────────────────────

╭──────────────────────────────────────────────────── history ────────────────────────────────────────────────────╮
│ 5                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯